# NK models of evolution

Code examples from [Think Complexity, 2nd edition](http://greenteapress.com/wp/complexity2), Chapter 11

Copyright 2016 Allen Downey, [MIT License](http://opensource.org/licenses/MIT)

In [ ]:
from __future__ import print_function, division

%matplotlib inline
%precision 3

import warnings
warnings.filterwarnings('ignore')

import numpy as np
from numpy.random import rand, randint, permutation

from thinkstats2 import Cdf
from thinkstats2 import RandomSeed

import thinkplot

from matplotlib import rc
rc('animation', html='html5')

## The NK landscape

Here's an implementation of an NK landscape.

A location in the landscape is represented by a NumPy array of N 0s and 1s.

The `index` attribute of `NKLandscape` is an array of indices into a location, which is an efficient way to select the overlapping slices.

The `cache` attribute is a dictionary that maps from `(i, slice)` to a fitness, where `i` indicates which of the `N` functions we want to evaluate and `slice` is the parameters of the function.

The first time we see a particular `(i, slice)` pair, we generate a random fitness value.  The we store it in the `cache` in case we need it again.

In [ ]:
class NKLandscape:
    def __init__(self, N, K, A=2):
        """Create a landscape.
        
        N: number of attributes
        K: number of interactions
        
        index is a NumPy array that specifies the interactions between
        attributes. (The K+1 entries in each row identify the attributes
        that contribute to the fitness of the row attribute.)
        """
        self.N = N
        self.K = K
        self.A = A
        self.cache = dict()
        
        iseq = np.array(range(N))
        jseq = np.array(range(K+1))
        self.index = (iseq[:, None] + jseq[None, :]) % N

    def random_loc(self):
        """Choose a random location."""
        # in NumPy version 1.11 we can pass dtype to randint
        return randint(self.A, size=self.N).astype(np.int8)
    
    def lookup(self, i, row):
        """Look up `row` in function `i`.
        
        i: int from 0 to N-1
        row: array of K+1 0s and 1s
        
        returns: f_i(row)
        """
        key = i, tuple(row)
        return self.cache.setdefault(key, rand())
    
    def fitness(self, loc):
        """Evaluates the fitness of a location.
        
        loc: array of N 0s and 1s
        
        returns: float fitness
        """
        slices = loc[self.index]
        fs = [self.lookup(i, row) for i, row in enumerate(slices)]
        return np.mean(fs)

Here's a small example.  The index shows how the traits are linked.  Trait 0 is linked to traits 1 and 2.  Trait 1 is linked to traits 2 and 3, etc.

In [ ]:
nkland = NKLandscape(5, 2)
nkland.index

Here's an example that evaluates the fitness at a random location:

In [ ]:
loc = nkland.random_loc()
loc, nkland.fitness(loc)

Here's what the landscape cache looks like after one evaluation:

In [ ]:
nkland.cache

If we evaluate the same location again, we should get the same value.

In [ ]:
loc, nkland.fitness(loc)

And if we evaluate a different location, we should get a different value.

In [ ]:
loc = nkland.random_loc()
loc, nkland.fitness(loc)

## The agents

Here's a parent class, `NKAgent`, that contains code used by all agents:

In [ ]:
class NKAgent:
    """Represents an agent in an NK model."""
    
    def __init__(self, landscape):
        """Create an agent at the given location.
        
        loc: array of N 0s and 1s
        landscape: reference to an NKLandscape
        """
        self.landscape = landscape
        self.loc = landscape.random_loc()
        self.fitness = landscape.fitness(self.loc)
        self.moves = 0
        self.done = False
        
    def mutation(self, direction):
        """Computes the location in the given direction.
        
        Result differs from the current location along the given axis.
        
        direction: int index from 0 to N-1
        
        returns: new array of N 0s and 1s
        """
        new_loc = self.loc.copy()
        new_loc[direction] ^= 1
        return new_loc
    
    def generate_mutations(self):
        """Generates all possible mutations from current location, in random order.
        
        yields: new array of N 0s and 1s
        """
        for direction in permutation(self.landscape.N):
            new_loc = self.mutation(direction)
            yield new_loc

    def consider(self, new_loc):
        """Moves if the fitness at `new_loc` is greater than or equal to current fitness.
        
        new_loc: array of N 0s and 1s
        
        return: True if the agent moved
        """
        new_fitness = self.landscape.fitness(new_loc)
        if new_fitness >= self.fitness:
            self.loc = new_loc
            self.fitness = new_fitness
            self.moves += 1
            return True
        return False
    
    def step(self):
        """Consider at least one mutation.
        
        Child classes should override this method.
        
        """
        pass

Here's an example using the NKAgent parent class.

In [ ]:
agent = NKAgent(nkland)
agent.loc, agent.fitness

We can choose a random direction.

In [ ]:
direction = randint(nkland.N)
new_loc = agent.mutation(direction)
new_loc

And consider moving.

In [ ]:
agent.consider(new_loc)

In [ ]:
agent.loc, agent.fitness

The following loop considers every direction, in random order, and accepts the first acceptable move.

In [ ]:
for direction in permutation(nkland.N):
    new_loc = agent.mutation(direction)
    if agent.consider(new_loc):
        print(agent.loc, agent.fitness)

Now we can encapsulate that strategy, called the "fitter" strategy, in a class:

In [ ]:
class NKAgentFitter(NKAgent):
    def step(self):
        """Consider all mutations in order and choose the first improvement."""
        if self.done:
            return False
        
        for new_loc in self.generate_mutations():
            if self.consider(new_loc):
                return True
            
        self.done = True
        return False

In [ ]:
agent = NKAgentFitter(nkland)
agent.loc, agent.fitness

In [ ]:
agent.step()

In [ ]:
agent.loc, agent.fitness

**Exercise:** Implement the other strategies described by Vidgen and Padget in [Sendero](http://jasss.soc.surrey.ac.uk/12/4/8.html)

Write a class definition called `NKAgentMutant` that implements the one-mutant neighbor strategy and a class definition called `NKAgentGreedy` that implements the greedy strategy.  Instantiate one of each and invoke their step methods.

In [ ]:
# Solution

class NKAgentMutant(NKAgent):

    def step(self):
        """Choose a mutation at random and consider it."""
        direction = np.random.randint(len(self.loc))
        new_loc = self.mutation(direction)
        return self.consider(new_loc)
            
class NKAgentGreedy(NKAgent):
    def step(self):
        """Consider all mutations and choose the best."""
        fits = [(self.landscape.fitness(new_loc), rand(), new_loc)
               for new_loc in self.generate_mutations()]
        fitness, _, new_loc = max(fits)
        return self.consider(new_loc)

## The simulator

Here's a class that runs simulations:

In [ ]:
class NKSimulation:
    
    def __init__(self, landscape, num_agents, agent_maker):
        """Create the simulation:
        
        landscape: NKLandscape
        num_agents: int number of agents
        agent_maker: function that makes agents
        """
        self.landscape = landscape
        self.agents = [agent_maker(landscape) for _ in range(num_agents)]
        
    def step(self):
        """Run step on each agent.
        
        returns: list of boolean, whether each agent moves
        """
        return [agent.step() for agent in self.agents]
        
    def get_fitnesses(self):
        """Returns a list of agent fitnesses."""
        return [agent.fitness for agent in self.agents]
    
    def get_locations(self):
        """Returns a list of agent locations."""
        return [agent.loc for agent in self.agents]
    
    def get_peaks(self):
        """Returns the set of unique locations.
        
        If all agents have run until they reach a local peak,
        the result is a subset of the peaks.
        """
        locs = [tuple(loc) for loc in self.get_locations()]
        return set(locs)

    def get_peak_heights(self):
        """Returns the set of unique heights.
        
        If all agents have run until they reach a local peak,
        the result is heights of a subset of the peaks.
        """
        return set(self.get_fitnesses())
    
    def get_path_lengths(self):
        """Returns the number of moves for each agent.
        
        If all agents have run until they reach a local peak,
        the result is the sequence of path lengths.
        """
        return [agent.moves for agent in self.agents]

And here's an example with small values of `N` and `K`.  Initially the distribution of fitness is centered around 0.5

In [ ]:
nkl = NKLandscape(5, 2)
nksim = NKSimulation(nkl, 100, NKAgentFitter)
cdf_fitness = Cdf(nksim.get_fitnesses())
thinkplot.Cdf(cdf_fitness)

cdf_fitness.Mean()

Here's how the distribution of fitness evolves after each step.

In [ ]:
thinkplot.PrePlot(7)

nksim = NKSimulation(nkl, 100, NKAgentFitter)

for i in range(7):
    cdf_fitness = Cdf(nksim.get_fitnesses())
    thinkplot.Cdf(cdf_fitness)
    nksim.step()

After every agent has found a peak, we can plot the distribution of fitness.

In [ ]:
cdf_fitness = Cdf(nksim.get_fitnesses())
thinkplot.Cdf(cdf_fitness)

The number of unique locations is a lower bound on the number of peaks.

In [ ]:
peaks = nksim.get_peaks()
len(peaks)

And we can look at the heights of the peaks.

In [ ]:
heights = nksim.get_peak_heights()
heights

Here's the distribution of path lengths.  A few agents are born on a peak.  The longest path is probably 5 or fewer.

In [ ]:
cdf_moves = Cdf(nksim.get_path_lengths())
thinkplot.Cdf(cdf_moves)

This function encapsulates the steps for running a simulation:

In [ ]:
def run_simulation(N=5, K=2, num_agents=100, agent_maker=NKAgentFitter):
    """Create a simulation and run until all agents find a peak.
    
    N: int number of traits
    K: int number of interactions
    num_agents: int number of agents
    agent_maker: function that creates agents
    
    returns NKSimulation object
    """
    nkl = NKLandscape(N, K)
    nksim = NKSimulation(nkl, num_agents, agent_maker)
    for i in range(100):
        steps = nksim.step()
        if np.sum(steps) == 0:
            break
    return nksim

Here's the same small example:

In [ ]:
nksim = run_simulation(N=5, K=2)

This function takes a completed simulation and summarizes the results.

In [ ]:
def plot_results(nksim):
    """Summarizes simulation results.
    
    nksim: NKSimulation object
    """
    cdf_fitness = Cdf(nksim.get_fitnesses())
    peaks = nksim.get_peaks()
    heights = nksim.get_peak_heights()
    cdf_moves = Cdf(nksim.get_path_lengths())
    
    print("Mean fitness: ", cdf_fitness.Mean())
    print("Mean path length: ", cdf_moves.Mean())
    print("Number of peaks:", len(peaks))

    thinkplot.PrePlot(cols=2)
    thinkplot.Cdf(cdf_fitness)
    thinkplot.Config(xlabel='Fitness', ylabel='CDF')

    thinkplot.SubPlot(2)
    thinkplot.Cdf(cdf_moves)
    thinkplot.Config(xlabel='Path Length', ylabel='CDF')

And here are the result with small values of `N` and `K`.

In [ ]:
plot_results(nksim)

Here's a simulation run with larger `N` and `K`:

In [ ]:
%time nksim = run_simulation(N=40, K=20)
plot_results(nksim)

**Exercise:** Starting with `N=5` and `K=3`, run simulations with increasing values of `N`, keeping `K` constant, and plot the number of peaks and mean path length as a function of `N`.

In [ ]:
# Solution

K = 3
res = []

ns = range(5, 30)
for N in ns:
    nksim = run_simulation(N, K)
    num_peaks = len(nksim.get_peaks())
    mean_path = np.mean(nksim.get_path_lengths())
    res.append((num_peaks, mean_path))
    
peaks, paths = np.transpose(res)

In [ ]:
# Solution

thinkplot.Plot(ns, peaks)
thinkplot.Config(xlabel='N', ylabel='Number of peaks')

In [ ]:
# Solution

thinkplot.Plot(ns, paths)
thinkplot.Config(xlabel='N', ylabel='Mean path length')

**Exercise:** Starting with `N=20` and `K=0`, run simulations with increasing values of `K`, keeping `N` constant, and plot the number of peaks and mean path length as a function of `K`.

In [ ]:
# Solution

N = 20
res = []

ks = range(0, 20)
for K in ks:
    nksim = run_simulation(N, K)
    num_peaks = len(nksim.get_peaks())
    mean_path = np.mean(nksim.get_path_lengths())
    res.append((num_peaks, mean_path))
    
peaks, paths = np.transpose(res)

In [ ]:
# Solution

thinkplot.Plot(ks, peaks)
thinkplot.Config(xlabel='N', ylabel='Number of peaks')

In [ ]:
# Solution

thinkplot.Plot(ks, paths)
thinkplot.Config(xlabel='N', ylabel='Mean path length')